> 在线讲义：[第 26 章 · 轨迹优化与 MPC：写成 QP（稠密 / 稀疏）、iLQR 的回传、MPPI 的路径积分](https://wikieden.github.io/math-and-eval-stats/lessons/26-trajopt-mpc.html) · 本书《数学与实验统计》

# 第 26 章 · 轨迹优化与 MPC：写成 QP（稠密 / 稀疏）、iLQR 的回传、MPPI 的路径积分

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/26-trajopt-mpc.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 稠密 QP | H U = −f x₀ | `solve_unconstrained()` | u₀ = −1.5871 |
| 第二步 盒约束 | L-BFGS-B | `solve_box(1.0)` | [−1, −1, −0.5396] |
| 第三步 稀疏 | 非零元计数 | `nnz_counts(50)` | 2500 vs 249 |
| 第四步 iLQR | 往回推 + 线搜索 | `ilqr()` | 238.78 → 80.32 |
| 第五步 MPPI | 指数加权 | `mppi_one_step(lam=0.1)` | −0.9525 |

例子：第 25 章的标量系统 x_{k+1} = 1.2·x_k + u_k，q = r = 1，终端代价 1，N = 10 步，x0 = 2。
iLQR 用一个编的非线性标量系统 x_{k+1} = x_k + 0.1·(sin x_k + u_k)；MPPI 用一步的标量问题，最优分布有闭式。
里程碑 ME-M20 的"一维小车 MPC 的稀疏 QP"和"一维小车上的 iLQR"留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
from scipy.optimize import minimize
# 下面几段来自 ch25_lqr_riccati.py（第 25 章）
import numpy as np
A = 1.2

B = 1.0

Q = 1.0

R = 1.0

def riccati_step(p: float, a: float = A, b: float = B, q: float = Q, r: float = R) -> tuple[float, float]:
    """动态规划往回推一步：已知下一步的值函数 p·x²，求这一步的 (p, K)。
    min_u [q x² + r u² + p (a x + b u)²] → u = −K x，K = a b p / (r + b² p)，新 p = q + a² p − (a b p)² / (r + b² p)。"""
    k = a * b * p / (r + b * b * p)
    return q + a * a * p - (a * b * p) ** 2 / (r + b * b * p), k

def finite_horizon(n: int, qf: float = 1.0, **kw) -> tuple[list[float], list[float]]:
    """从终端 p_N = q_f 往回推 n 步，返回按时间顺序的 [p_0, …, p_N] 和 [K_0, …, K_{N−1}]。"""
    ps, ks, p = [qf], [], qf
    for _ in range(n):
        p, k = riccati_step(p, **kw)
        ps.append(p)
        ks.append(k)
    return ps[::-1], ks[::-1]

N, X0 = 10, 2.0

**函数 `dense_qp`**：把状态消掉：X = S_x x0 + S_u U，代价 = Uᵀ H U + 2 x0 fᵀ U + 常数。返回 (H, f, S_x, S_u)。

In [ ]:
def dense_qp(n: int = N, a: float = A, b: float = B, q: float = 1.0, r: float = 1.0, qf: float = 1.0):
    """把状态消掉：X = S_x x0 + S_u U，代价 = Uᵀ H U + 2 x0 fᵀ U + 常数。返回 (H, f, S_x, S_u)。"""
    sx = np.array([a ** (k + 1) for k in range(n)])
    su = np.zeros((n, n))
    for k in range(n):
        for j in range(k + 1):
            su[k, j] = a ** (k - j) * b
    w = np.full(n, q)
    w[-1] = qf
    h = su.T @ (w[:, None] * su) + r * np.eye(n)
    f = su.T @ (w * sx)
    return h, f, sx, su

**函数 `solve_unconstrained`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def solve_unconstrained(x0: float = X0) -> np.ndarray:
    h, f, _, _ = dense_qp()
    return np.linalg.solve(h, -f * x0)

**函数 `solve_box`**：加上 |u| ≤ umax 的盒约束，用 L-BFGS-B 解这个 QP。

In [ ]:
def solve_box(umax: float, x0: float = X0) -> np.ndarray:
    """加上 |u| ≤ umax 的盒约束，用 L-BFGS-B 解这个 QP。"""
    h, f, _, _ = dense_qp()
    res = minimize(lambda u: u @ h @ u + 2 * x0 * f @ u, np.zeros(N), jac=lambda u: 2 * h @ u + 2 * x0 * f,
                   bounds=[(-umax, umax)] * N, method="L-BFGS-B", options={"ftol": 1e-14, "gtol": 1e-12})
    return res.x

**函数 `nnz_counts`**：标量系统：稠密形式 H 是 n×n 满矩阵；稀疏形式变量 [x_1..x_n, u_0..u_{n−1}]，

In [ ]:
def nnz_counts(n: int) -> tuple[int, int]:
    """标量系统：稠密形式 H 是 n×n 满矩阵；稀疏形式变量 [x_1..x_n, u_0..u_{n−1}]，
    Hessian 对角 2n 个非零，动力学等式 x_{k+1} − a x_k − b u_k = 0 每行最多 3 个非零（第一行 x_0 已知，只有 2 个）。"""
    return n * n, 2 * n + (3 * n - 1)

In [ ]:
DT = 0.1

**函数 `f_nl`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def f_nl(x, u):
    return x + DT * (np.sin(x) + u)

**函数 `ilqr`**：iLQR：沿当前轨迹线性化（A_k = 1 + 0.1 cos x_k，B = 0.1）、二次化代价，往回推得到修正量，带线搜索往前滚。返回每轮的总代价；传入 trajs 列表时顺便把每轮的状态轨迹存进去（画图用）。

In [ ]:
def ilqr(x0: float = 2.0, n: int = 30, q: float = 1.0, r: float = 1.0, iters: int = 8, trajs: list | None = None) -> list[float]:
    """iLQR：沿当前轨迹线性化（A_k = 1 + 0.1 cos x_k，B = 0.1）、二次化代价，往回推得到修正量，带线搜索往前滚。返回每轮的总代价；传入 trajs 列表时顺便把每轮的状态轨迹存进去（画图用）。"""
    u = np.zeros(n)

    def roll(us):
        xs = [x0]
        for v in us:
            xs.append(f_nl(xs[-1], v))
        return np.array(xs)

    def cost(xs, us):
        return float(q * (xs ** 2).sum() + r * (us ** 2).sum())

    xs = roll(u)
    hist = [cost(xs, u)]
    if trajs is not None:
        trajs.append(xs)
    for _ in range(iters):
        vx, vxx = 2 * q * xs[-1], 2 * q
        kff, kfb = np.zeros(n), np.zeros(n)
        for k in range(n - 1, -1, -1):
            ak, bk = 1 + DT * np.cos(xs[k]), DT
            qx = 2 * q * xs[k] + ak * vx
            qu = 2 * r * u[k] + bk * vx
            qxx = 2 * q + ak * vxx * ak
            quu = 2 * r + bk * vxx * bk
            qux = bk * vxx * ak
            kff[k], kfb[k] = -qu / quu, -qux / quu
            vx = qx + kfb[k] * quu * kff[k] + kfb[k] * qu + qux * kff[k]
            vxx = qxx + kfb[k] * quu * kfb[k] + 2 * kfb[k] * qux
        best = None
        for alpha in (1.0, 0.5, 0.25, 0.1):
            xn, un = [x0], np.zeros(n)
            for k in range(n):
                un[k] = u[k] + alpha * kff[k] + kfb[k] * (xn[-1] - xs[k])
                xn.append(f_nl(xn[-1], un[k]))
            xn = np.array(xn)
            if cost(xn, un) < hist[-1]:
                best = (xn, un)
                break
        if best is None:
            break
        xs, u = best
        hist.append(cost(xs, u))
        if trajs is not None:
            trajs.append(xs)
    return hist

**函数 `mppi_one_step`**：一步问题：u ~ N(0, σ²) 是“先验”控制分布，代价 S(u) = (x0 + u)²。

In [ ]:
def mppi_one_step(x0: float = 1.0, sigma: float = 1.0, lam: float = 1.0, n: int = 100000, seed: int = 0):
    """一步问题：u ~ N(0, σ²) 是“先验”控制分布，代价 S(u) = (x0 + u)²。
    最优分布 q*(u) ∝ N(u; 0, σ²)·exp(−S(u)/λ) 还是高斯：精度 1/σ² + 2/λ，均值 −(2x0/λ)/(1/σ² + 2/λ)。
    MPPI 用样本加权 w ∝ exp(−S/λ) 估计这个均值。返回 (闭式均值, 加权估计, 有效样本数)。"""
    rng = np.random.default_rng(seed)
    u = rng.normal(0, sigma, n)
    s = (x0 + u) ** 2
    w = np.exp(-(s - s.min()) / lam)
    w /= w.sum()
    exact = -(2 * x0 / lam) / (1 / sigma ** 2 + 2 / lam)
    return exact, float((w * u).sum()), float(1 / (w ** 2).sum())

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 稠密 QP：把状态消掉，只剩 N 个控制量")
u = solve_unconstrained()
_, ks = finite_horizon(N)
print(f"  QP 解出的 u_0 = {u[0]:.4f}；有限时域 Riccati 的 −K_0·x0 = {-ks[0] * X0:.4f}（同一个最优解）")
h, f, _, _ = dense_qp()
print(f"  H 是 {h.shape[0]}×{h.shape[0]} 满矩阵，条件数 {np.linalg.cond(h):.1f}")

In [ ]:
print("\n第二步 加约束 |u| ≤ 1：Riccati 写不出来，QP 照样解")
ub = solve_box(1.0)
print(f"  无约束 u_0..u_2 = {np.round(u[:3], 4).tolist()}；有约束 = {np.round(ub[:3], 4).tolist()}")
print("  MPC：每个控制周期只执行 u_0，下一周期用新测量的 x 再解一次")

In [ ]:
print("\n第三步 稀疏形式：保留状态当变量，矩阵变大但几乎全是 0")
for n in (10, 50, 200):
    d, s = nnz_counts(n)
    print(f"  N = {n}：稠密 H 非零元 {d}，稀疏 Hessian + 等式约束非零元 {s}")

In [ ]:
print("\n第四步 iLQR：非线性系统，反复“线性化 + 二次化 + 往回推 + 往前滚”")
hist = ilqr()
print(f"  每轮总代价：{[round(c, 4) for c in hist]}")

In [ ]:
print("\n第五步 MPPI：最优控制分布 ∝ 先验 × exp(−代价/λ)")
ex, est, ess = mppi_one_step()
print(f"  x0 = 1、σ = 1、λ = 1：闭式最优均值 {ex:.4f}，10 万样本加权估计 {est:.4f}，有效样本数 {ess:.0f}")
ex2, est2, ess2 = mppi_one_step(lam=0.1)
print(f"  λ = 0.1：闭式 {ex2:.4f}，估计 {est2:.4f}，有效样本数 {ess2:.0f}（λ 越小越贪心、有效样本越少）")

## 3. 代码里才看得到的

1. **盒约束 QP 用 L-BFGS-B 解**：它原生支持变量上下界；真正的 MPC 会用 OSQP 这类专门的 QP 求解器，速度快几个数量级。
2. **iLQR 的线搜索从 α = 1 往下试**：代价没下降就把步长减半，4 次都不行就停（这个例子每轮第一次就成功）。
3. **MPPI 权重先减最小代价**：e^{−S/λ} 在 λ 小时会下溢成 0，减掉 min S 后最好的那条权重是 1，比例不变。

## 4. 自己改一改

1. 把 `solve_box` 的限位改成 0.5，看前几步贴着限位。
2. 把 iLQR 的 x₀ 改成 3（离 sin x 的另一个平衡点 π 很近），看收敛到哪、要几轮。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 只记油门是打靶，稠密而病态；连状态一起记是转录，稀疏又稳当；
> 无约束时 QP 和 Riccati 同解，加了限位只能靠 QP；
> MPC 每拍解一次、只走第一步；
> 非线性就 iLQR：线性化、往回推、往前滚；
> MPPI：先验乘 e 的负代价除 λ。